# Cifrium — Early Churn Detection at Day 14

### Product Analytics + Machine Learning

**Business problem:** the course has four modules, and the largest retention loss occurs between modules 1 and 2.

**North Star Metric:** Course Completion Rate.

**Model decision point:** day 14 after enrollment.

**Output:** Churn Risk Score used to prioritize early interventions.

This notebook is organized as a product case, not as a model leaderboard:

1. diagnose retention;
2. define the intervention point;
3. build point-in-time behavioral features;
4. quantify early behavioral signals;
5. train and validate churn models on future cohorts;
6. translate model quality into operational targeting;
7. define the experiment that can prove business impact.


In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.calibration import calibration_curve

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.features import build_feature_matrix, assert_no_forbidden_features, normalize_id
from src.evaluation import evaluate_scores, gains_table

DATA = ROOT / "data"
RANDOM_STATE = 42
OBSERVATION_DAYS = 14
CURATOR_CAPACITY = 0.20


## 1. Load data

The project uses end-state module tables for funnel / outcome analysis and timestamped
raw tables for D14 behavioral features.


In [ ]:
files = {
    "stats_m1": "stats__module_1.csv",
    "stats_m2": "stats__module_2.csv",
    "stats_m3": "stats__module_3.csv",
    "stats_m4": "stats__module_4.csv",
    "activity": "user_activity_histories.csv",
    "user_lessons": "user_lessons.csv",
    "answers": "user_answers.csv",
    "media": "wk_media_view_sessions.csv",
}

missing = [name for name, f in files.items() if not (DATA / f).exists()]
if missing:
    raise FileNotFoundError(
        "Missing source files in data/: " + ", ".join(missing)
    )

data = {name: pd.read_csv(DATA / f) for name, f in files.items()}

for name, df in data.items():
    print(f"{name:15s} {df.shape[0]:>9,} rows × {df.shape[1]:>3} cols")


# Part I. Product analytics

## 2. Course funnel and retention diagnosis

Before building a model, we need to establish where the business problem is concentrated.

The module tables contain final module outcomes, so they are appropriate for funnel
diagnosis but **not** for D14 predictive features.


In [ ]:
module_tables = {
    1: data["stats_m1"],
    2: data["stats_m2"],
    3: data["stats_m3"],
    4: data["stats_m4"],
}

funnel_rows = []
for module, df in module_tables.items():
    tmp = df.copy()
    status = tmp["Статус"].astype("string")
    funnel_rows.append({
        "module": module,
        "students": tmp["user_id"].nunique(),
        "completed": (status == "Завершил").sum(),
        "churned": (status == "Отчислен").sum(),
        "completion_rate": (status == "Завершил").mean(),
        "churn_rate": (status == "Отчислен").mean(),
    })

funnel = pd.DataFrame(funnel_rows)
funnel


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(funnel["module"].astype(str), funnel["students"])
plt.xlabel("Module")
plt.ylabel("Students")
plt.title("Student funnel by module")
plt.show()

funnel[["module", "students", "completion_rate", "churn_rate"]]


### Product interpretation

The goal of this step is to separate two questions:

- **Where is the largest business loss?**
- **Where do we still have enough time to influence the outcome?**

For this course, the first module is the natural intervention zone. Downstream completion
remains the North Star, but the **M1 → M2 transition** is a faster decision metric for
testing retention interventions.


## 3. Cohort-level retention

A single average can hide deterioration or improvement over time. We therefore monitor
retention by enrollment cohort.


In [ ]:
m1 = data["stats_m1"].copy()
m1["Дата зачисления"] = pd.to_datetime(m1["Дата зачисления"], errors="coerce")
m1["cohort_month"] = m1["Дата зачисления"].dt.to_period("M").astype(str)
m1["churn"] = (m1["Статус"] == "Отчислен").astype(int)

cohort_retention = (
    m1.groupby("cohort_month")
    .agg(
        students=("user_id", "nunique"),
        churn_rate=("churn", "mean"),
    )
    .reset_index()
)
cohort_retention["retention_rate"] = 1 - cohort_retention["churn_rate"]
cohort_retention


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cohort_retention["cohort_month"], cohort_retention["retention_rate"], marker="o")
plt.xticks(rotation=45)
plt.ylabel("Module 1 retention rate")
plt.xlabel("Enrollment cohort")
plt.title("Retention by enrollment cohort")
plt.tight_layout()
plt.show()


## 4. Product metric tree

**North Star:** Course Completion Rate

Leading metrics:
- M1 → M2 transition rate;
- D14 active-day rate;
- D14 inactivity / recency;
- D14 task engagement;
- D14 content consumption.

The model score is a **decision-support metric**, not a North Star.


# Part II. Point-in-time behavioral analytics

## 5. Build the D14 analytical snapshot

Every feature is calculated from events available no later than day 14.

If a raw source does not contain a usable event timestamp, its behavioral aggregate is
not used automatically.


In [ ]:
features = build_feature_matrix(
    stats_m1=data["stats_m1"],
    activity=data["activity"],
    user_lessons=data["user_lessons"],
    answers=data["answers"],
    media=data["media"],
    days=OBSERVATION_DAYS,
)

features.shape, features.head()


In [ ]:
assert_no_forbidden_features(features.columns)

behavior_cols = [c for c in features.columns if c.endswith("_d14")]
print("D14 behavioral features:", len(behavior_cols))
print(*behavior_cols, sep="\n")


## 6. Define the prediction target

The operational target is **future churn after the D14 observation window**.

For the available course export, the first-module final status is used as the matured
outcome label:

- `1` — churn / expelled;
- `0` — completed module 1.

The status itself never enters the feature matrix.


In [ ]:
labels = data["stats_m1"][["user_id", "Статус", "Дата зачисления"]].copy()
labels["user_id"] = normalize_id(labels["user_id"])
labels["Дата зачисления"] = pd.to_datetime(labels["Дата зачисления"], errors="coerce")
labels["churn"] = labels["Статус"].map({"Отчислен": 1, "Завершил": 0})
labels = labels.dropna(subset=["churn", "Дата зачисления"]).drop_duplicates("user_id")
labels["churn"] = labels["churn"].astype(int)

df = features.merge(
    labels[["user_id", "churn", "Дата зачисления"]],
    on=["user_id", "Дата зачисления"],
    how="inner",
)

print(df["churn"].value_counts())
print("Churn rate:", round(df["churn"].mean(), 3))


## 7. Early behavioral differences

For product analytics, effect size matters more than dumping dozens of distributions.
We compare D14 behavior between future retained and churned students and rank features
by standardized mean difference.


In [ ]:
numeric_d14 = [c for c in behavior_cols if c in df.select_dtypes(include=np.number).columns]

rows = []
for c in numeric_d14:
    a = df.loc[df["churn"] == 0, c].astype(float)
    b = df.loc[df["churn"] == 1, c].astype(float)
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    smd = (b.mean() - a.mean()) / pooled if pooled and np.isfinite(pooled) else 0
    rows.append({
        "feature": c,
        "retained_mean": a.mean(),
        "churn_mean": b.mean(),
        "smd_churn_minus_retained": smd,
        "abs_smd": abs(smd),
    })

behavior_effects = pd.DataFrame(rows).sort_values("abs_smd", ascending=False)
behavior_effects.head(12)


In [ ]:
top = behavior_effects.head(10).sort_values("smd_churn_minus_retained")

plt.figure(figsize=(8, 5))
plt.barh(top["feature"], top["smd_churn_minus_retained"])
plt.axvline(0, linewidth=1)
plt.xlabel("Standardized mean difference: churn − retained")
plt.title("Strongest D14 behavioral differences")
plt.tight_layout()
plt.show()


### Product use of this analysis

This table answers a different question from feature importance:

> **Which early behaviors are visibly different enough to become intervention levers?**

Examples:
- high recency / inactivity → outreach trigger;
- weak content depth → content recommendation;
- low task attempts → study-plan support;
- zero active days → onboarding / access issue.

Predictive importance alone is not enough: the team needs signals that map to actions.


## 8. Behavioral risk curves

For the strongest continuous behaviors we examine empirical churn rate by behavior
quantile. This helps product teams see whether risk changes smoothly or has a practical
trigger zone.


In [ ]:
def risk_by_quantile(frame, feature, bins=5):
    tmp = frame[[feature, "churn"]].dropna().copy()
    tmp["bucket"] = pd.qcut(tmp[feature], q=bins, duplicates="drop")
    return (
        tmp.groupby("bucket", observed=True)
        .agg(students=("churn", "size"), churn_rate=("churn", "mean"))
        .reset_index()
    )

if len(behavior_effects):
    example_feature = behavior_effects.iloc[0]["feature"]
    curve = risk_by_quantile(df, example_feature)
    display(curve)

    plt.figure(figsize=(8, 4))
    plt.plot(range(len(curve)), curve["churn_rate"], marker="o")
    plt.xticks(range(len(curve)), curve["bucket"].astype(str), rotation=35, ha="right")
    plt.ylabel("Observed churn rate")
    plt.xlabel(example_feature)
    plt.title("Empirical D14 risk curve")
    plt.tight_layout()
    plt.show()


# Part III. Machine learning

## 9. Time-based validation

A random train/test split can mix students from the same acquisition / teaching period.
For a production-like estimate, we hold out the latest enrollment cohorts.


In [ ]:
df = df.sort_values("Дата зачисления").reset_index(drop=True)
split_idx = int(len(df) * 0.80)

train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()

drop_cols = {"user_id", "Дата зачисления", "churn"}
feature_cols = [c for c in df.columns if c not in drop_cols]
assert_no_forbidden_features(feature_cols)

categorical = [
    c for c in ["Уровень", "teacher_id", "id параллели"]
    if c in feature_cols
]
numeric = [c for c in feature_cols if c not in categorical]

X_train, y_train = train[feature_cols], train["churn"]
X_test, y_test = test[feature_cols], test["churn"]

print("Train:", train["Дата зачисления"].min(), "→", train["Дата зачисления"].max(), len(train))
print("Test: ", test["Дата зачисления"].min(), "→", test["Дата зачисления"].max(), len(test))


## 10. Baseline: Logistic Regression

The baseline is useful for calibration, stability and interpretability.


In [ ]:
prep_linear = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])

logit = Pipeline([
    ("prep", prep_linear),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

logit.fit(X_train, y_train)
p_logit = logit.predict_proba(X_test)[:, 1]
evaluate_scores(y_test, p_logit, CURATOR_CAPACITY)


## 11. Non-linear sklearn model

HistGradientBoosting captures interactions between behavioral signals while keeping the
pipeline reproducible with standard sklearn.


In [ ]:
# HistGradientBoosting needs numeric input. Categorical IDs are excluded here;
# behavioral features remain the primary signal.
hgb_numeric = [c for c in numeric if c in X_train.columns]

hgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    )),
])

hgb.fit(X_train[hgb_numeric], y_train)
p_hgb = hgb.predict_proba(X_test[hgb_numeric])[:, 1]
evaluate_scores(y_test, p_hgb, CURATOR_CAPACITY)


## 12. Strong tabular model: CatBoost

CatBoost handles non-linear interactions and categorical context well. It is trained
with a validation tail from the historical training period for early stopping.


In [ ]:
catboost_available = True
try:
    from catboost import CatBoostClassifier
except ImportError:
    catboost_available = False
    print("CatBoost is not installed; skipping this model.")

p_cat = None
cat_model = None

if catboost_available:
    Xtr = X_train.copy()
    Xte = X_test.copy()

    for c in categorical:
        Xtr[c] = Xtr[c].astype("string").fillna("missing")
        Xte[c] = Xte[c].astype("string").fillna("missing")
    for c in numeric:
        Xtr[c] = pd.to_numeric(Xtr[c], errors="coerce")
        Xte[c] = pd.to_numeric(Xte[c], errors="coerce")

    val_idx = int(len(Xtr) * 0.85)
    X_fit, X_val = Xtr.iloc[:val_idx], Xtr.iloc[val_idx:]
    y_fit, y_val = y_train.iloc[:val_idx], y_train.iloc[val_idx:]

    cat_model = CatBoostClassifier(
        iterations=1500,
        depth=6,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=RANDOM_STATE,
        verbose=False,
        l2_leaf_reg=5,
    )
    cat_model.fit(
        X_fit, y_fit,
        cat_features=categorical,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False,
    )
    p_cat = cat_model.predict_proba(Xte)[:, 1]
    display(pd.Series(evaluate_scores(y_test, p_cat, CURATOR_CAPACITY)))


## 13. Model comparison

The selected model should perform well both statistically and operationally.


In [ ]:
rows = []
for name, score in [
    ("Logistic Regression", p_logit),
    ("HistGradientBoosting", p_hgb),
    ("CatBoost", p_cat),
]:
    if score is not None:
        row = {"model": name}
        row.update(evaluate_scores(y_test, score, CURATOR_CAPACITY))
        rows.append(row)

model_results = pd.DataFrame(rows).sort_values(
    ["recall_at_k", "pr_auc", "roc_auc"],
    ascending=False,
)
model_results


In [ ]:
best_name = model_results.iloc[0]["model"]
score_map = {
    "Logistic Regression": p_logit,
    "HistGradientBoosting": p_hgb,
    "CatBoost": p_cat,
}
best_score = score_map[best_name]

print("Selected model:", best_name)


## 14. Gains / lift: the product view of ranking quality

If the first risk decile has a lift of 3×, the top 10% of students contain churners at
three times the cohort baseline rate. This is more directly actionable than accuracy.


In [ ]:
gains = gains_table(y_test, best_score, bins=10)
gains


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(gains["decile"].astype(str), gains["lift"])
plt.axhline(1, linestyle="--")
plt.xlabel("Risk decile (1 = highest risk)")
plt.ylabel("Lift vs cohort churn rate")
plt.title(f"Churn lift by risk decile — {best_name}")
plt.show()


## 15. Calibration

Risk scores are useful for prioritization even before perfect calibration, but a
probability shown to operations should be calibrated.


In [ ]:
prob_true, prob_pred = calibration_curve(y_test, best_score, n_bins=8, strategy="quantile")

plt.figure(figsize=(5, 5))
plt.plot(prob_pred, prob_true, marker="o", label=best_name)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Predicted risk")
plt.ylabel("Observed churn")
plt.title("Calibration curve")
plt.legend()
plt.show()


## 16. Feature importance for intervention design

For CatBoost we inspect model importance, but interpretation stays behavioral:
importance tells us what improves ranking; product analytics tells us what can be acted on.


In [ ]:
if cat_model is not None and best_name == "CatBoost":
    importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": cat_model.get_feature_importance(),
    }).sort_values("importance", ascending=False)

    display(importance.head(15))

    plot_imp = importance.head(12).sort_values("importance")
    plt.figure(figsize=(8, 5))
    plt.barh(plot_imp["feature"], plot_imp["importance"])
    plt.xlabel("CatBoost feature importance")
    plt.title("Top D14 risk signals")
    plt.tight_layout()
    plt.show()


# Part IV. From score to product action

## 17. Capacity-aware risk segmentation

The score is converted into an operations queue.

Example:
- top 10% → High;
- next 10% → Medium;
- remaining 80% → Low.

The exact cutoffs should follow curator capacity and measured intervention ROI.


In [ ]:
scored = test[["user_id", "churn"]].copy()
scored["churn_risk_score"] = best_score
scored["risk_rank_pct"] = scored["churn_risk_score"].rank(pct=True, method="first")

scored["risk_segment"] = np.select(
    [
        scored["risk_rank_pct"] >= 0.90,
        scored["risk_rank_pct"] >= 0.80,
    ],
    ["High", "Medium"],
    default="Low",
)

segment_summary = (
    scored.groupby("risk_segment")
    .agg(
        students=("user_id", "size"),
        avg_risk=("churn_risk_score", "mean"),
        observed_churn=("churn", "mean"),
    )
    .reindex(["High", "Medium", "Low"])
)
segment_summary


## 18. Intervention playbook

| Early signal | Likely friction | Intervention |
|---|---|---|
| Zero / very low D14 activity | onboarding, access, motivation | personal outreach |
| Long inactivity gap | disengagement | reactivation message + curator follow-up |
| Few task attempts | learning difficulty / avoidance | study plan + office hours |
| Low video depth | content format friction | shorter recommended path / alternative material |
| High risk with normal activity | academic difficulty | targeted academic support |

The model decides **priority**. Behavioral diagnostics decide **what to do**.


## 19. Business impact scenario

An offline model has no causal business impact until an intervention works.

The scenario below converts targeting quality and intervention uplift into expected
incremental retained students.


In [ ]:
def intervention_scenario(
    cohort_size,
    contact_share,
    precision_at_k,
    uplift_pp,
):
    contacted = cohort_size * contact_share
    true_risk_contacted = contacted * precision_at_k
    incremental_retained = true_risk_contacted * uplift_pp
    return pd.Series({
        "cohort_size": cohort_size,
        "contacted": contacted,
        "true_risk_in_contacted": true_risk_contacted,
        "incremental_retained_students": incremental_retained,
        "retention_uplift_pp_on_cohort": incremental_retained / cohort_size,
    })

best_metrics = evaluate_scores(y_test, best_score, CURATOR_CAPACITY)

scenario = intervention_scenario(
    cohort_size=1000,
    contact_share=CURATOR_CAPACITY,
    precision_at_k=best_metrics["precision_at_k"],
    uplift_pp=0.10,  # replace with A/B-tested causal uplift
)
scenario


## 20. Experiment design

**Population:** students scored High Risk on D14.

**Randomization:** within eligible high-risk students.

**Treatment:** curator intervention.

**Control:** business-as-usual.

**Primary metric:** M1 → M2 transition rate.

**Secondary metric:** Course Completion Rate.

**Guardrails:** curator workload, complaint / opt-out rate, intervention cost.

### Why randomize after scoring?

The model answers **who is at risk**.

The experiment answers **whether our action changes the outcome**.

These are different questions and should be measured separately.


# Final recommendation

### Product

1. Make D14 risk scoring part of the Module 1 retention process.
2. Give curators a ranked queue, not a binary “churn / no churn” label.
3. Attach behavioral reasons to each high-risk student.
4. Start with one clearly defined intervention and randomize it.
5. Optimize for incremental M1 → M2 transition, then verify downstream Course Completion Rate.

### Analytics

Monitor weekly:
- D14 active-student share;
- D14 zero-activity share;
- high-risk population share;
- Recall@Top-20%;
- Precision@Top-20%;
- score calibration;
- intervention uplift;
- M1 → M2 transition;
- Course Completion Rate.

### ML

Retrain by cohort when enough new matured labels accumulate and monitor:
- ranking drift;
- calibration drift;
- feature distribution drift;
- cohort-level performance;
- subgroup stability.

The objective is not to maximize an isolated offline metric. It is to build a reliable
decision system that helps Cifrium intervene early enough to improve retention.
